# Imports

In [87]:
import os
import pickle
import re

# for testing purposes
import pandas as pd

# Preprocessing Lexicons

Here, we take the following steps:
- Read all the lexicon files
- Replace and clean up the special characters
- Express all lexicons in the following regular expression format: 
  ```
  <String Begining or whitespace>lexicon<Any character for any number of times><Line End or whitespace>
  ```

- Finally, we join all these individual regular expressions into one master regex usinf the or operator `("|")`

In [88]:
lexicons_dict = {}

In [89]:
def read_in_lexicons(directory):
    for filename in os.listdir(directory):
        with open(directory+filename, encoding = "mac_roman") as lexicons:
            if filename.startswith("."):
                continue
            lines = [r"\b" + line.replace("\n", "").replace("*", "") + r"\S*\b" for line in lexicons]
        clean_name = re.sub('.txt', '', filename)
        lexicons_dict[clean_name] = "|".join(lines)

In [90]:
read_in_lexicons(directory="lexicons/liwc_lexicons/") # Reads in LIWC Lexicons
read_in_lexicons(directory="lexicons/other_lexicons/") # Reads in Other Lexicons

# Saving Preprocessed Lexicons Dictionary

In [92]:
with open("lexicons_dict.pkl", "wb") as lexicons_pickle_file:
    pickle.dump(lexicons_dict, lexicons_pickle_file)

with open("lexicons_dict.pkl", "rb") as lexicons_pickle_file:
    lexicons_dict_loaded = pickle.load(lexicons_pickle_file)

lexicons_dict_loaded == lexicons_dict

True

# Testing Pipeline

In [93]:
data = {
    'message': [
        'hello',
        'hi', # for some reason this triggers really large values in things like 'bio' and 'cognitive mech'
        'hi!',
        'I am happy today', # first_person is always 0
        'i am good', # first_person is always 0
        '()()()()' # all symbols leads to the regex returning len(regex)
    ]
}

chat_df = pd.DataFrame(data)

In [94]:
pd.concat(
    # Finding the # of occurances of lexicons of each type for all the messages.
    [pd.DataFrame(chat_df["message"].apply(lambda chat: len(re.findall(chat, regex))))\
                                    .rename({"message": lexicon_type}, axis=1)\
        for lexicon_type, regex in lexicons_dict.items()], 
    axis=1
)


,discrepancies,hear,home,conjunction,certainty,inclusive,bio,achievement,adverbs,anxiety,...,auxiliary_verbs,cognitive_mech,preposition,first_person_plural,percept,second_person,positive_words,first_person,nltk_english_stopwords,hedge_words
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,1,0,1,1,0,17,1,0,1,...,0,11,2,0,11,1,38,0,6,1
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,1108,685,1299,354,1267,219,7686,2681,978,1322,...,1920,10623,769,148,3647,259,33135,117,2189,121
